<a href="https://colab.research.google.com/github/toxzak-svg/fabq-rc/blob/main/FABQ-RC-GGUF-Export.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FABQ-RC GGUF Export Pipeline

## Overview

This notebook exports FABQ-RC quantized weights to GGUF Q1_K format.

**Pipeline:**
1. Download FABQ-RC quantized model (.pth)
2. Save reconstructed weights as Q1_K GGUF
3. Upload PRODUCED files to HuggingFace

**Requirements:** A100 80GB, ~80GB disk

**Output Repo:** `toxzak/Qwen3.6-27B-FABQ-RC-GGUF`


In [ ]:
# Cell 1: Setup
import os, sys, subprocess, time

reqs = ['torch', 'numpy', 'huggingface_hub', 'transformers', 'safetensors', 'sentencepiece', 'protobuf', 'accelerate']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + reqs)

# Clone llama.cpp
LLAMA_CPP_DIR = '/content/llama.cpp'
if not os.path.exists(LLAMA_CPP_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ggerganov/llama.cpp.git', LLAMA_CPP_DIR], check=True)

# Mount drive for caching
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = '/content/drive/MyDrive/fabqrc_cache'
except:
    CACHE_DIR = '/content'
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs('/content/output', exist_ok=True)
print('Setup complete')

In [ ]:
# Cell 2: Download FABQ-RC model
from huggingface_hub import snapshot_download

HF_TOKEN = os.environ.get('HF_TOKEN', '')
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN') or HF_TOKEN
except:
    pass

print('Downloading FABQ-RC model...')
t0 = time.time()

model_dir = snapshot_download(
    'toxzak/Qwen3.6-27B-FABQ-RC',
    allow_patterns=['quantized_model.pth'],
    token=HF_TOKEN or None,
)

pth_path = os.path.join(model_dir, 'quantized_model.pth')
print(f'Downloaded: {os.path.getsize(pth_path)/1e9:.1f} GB in {time.time()-t0:.0f}s')

In [ ]:
# Cell 3: Load FABQ-RC state and reconstruct weights
import torch

print('Loading FABQ-RC state...')
state = torch.load(pth_path, map_location='cpu', weights_only=False)
print(f'Version: {state.get("version", "unknown")}')
print(f'Layers: {len(state.get("layers", {}))}')

def reconstruct_weights(state):
    """Reconstruct FP16 weights from FABQ-RC state."""
    layers = state.get('layers', {})
    tensors = []

    for name, ldata in layers.items():
        out_c = ldata['original_out_features']
        in_c = ldata['original_in_features']
        weight = torch.zeros(out_c, in_c, dtype=torch.float16)

        # int4 channels
        if len(ldata.get('int8_channels', [])) > 0:
            ch = ldata['int8_channels'].long()
            w = ldata['int8_weights'].to(torch.float16)
            s = ldata['int8_scales']
            if s.dim() == 1:
                s = s.unsqueeze(-1)
            weight[ch] = w * s

        # binary channels
        if len(ldata.get('binary_channels', [])) > 0:
            ch = ldata['binary_channels'].long()
            if 'binary_reconstructed_weights' in ldata:
                weight[ch] = ldata['binary_reconstructed_weights']

        tensors.append((name, weight))

        if ldata.get('bias') is not None:
            tensors.append((name + '.bias', ldata['bias'].cpu().half()))

    return tensors

print('Reconstructing weights...')
t0 = time.time()
fp16_tensors = reconstruct_weights(state)
print(f'Reconstructed {len(fp16_tensors)} tensors in {time.time()-t0:.0f}s')

In [ ]:
# Cell 4: Download base model for config/tokenizer
from transformers import AutoConfig, AutoTokenizer

BASE_MODEL = 'Qwen/Qwen3.6-27B'

print('Downloading config/tokenizer...')
config = AutoConfig.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
print(f'Config: {config.num_hidden_layers} layers, {config.hidden_size} hidden')

In [ ]:
# Cell 5: Write Q1_K GGUF (proper format)
# GGUF constants
GGUF_MAGIC = 0x46554747
GGUF_VERSION = 3
ALIGN = 32
Q1_K_ELEMS = 256
Q1_K_SIZE = 352

GGML_TYPE_F16 = 1
GGML_TYPE_Q1_K = 20

GGUF_TYPE_UINT32 = 4
GGUF_TYPE_FLOAT32 = 6
GGUF_TYPE_STRING = 8
GGUF_TYPE_UINT64 = 10

def align_to(x, a):
    return ((x + a - 1) // a) * a

def write_str(f, s):
    enc = s.encode('utf-8')
    f.write(struct.pack('<Q', len(enc)))
    f.write(enc)

def map_name(name):
    """Map FABQ-RC name to GGUF tensor name."""
    name = name.replace('model.', '')
    parts = name.split('.')

    if 'layers' in parts:
        layer_idx = parts[parts.index('layers') + 1]
        rest = parts[parts.index('layers') + 2:]

        if 'linear_attn' in rest:
            proj = rest[rest.index('linear_attn') + 1] if 'linear_attn' in rest else ''
            proj_map = {'q_proj': 'attn_q', 'k_proj': 'attn_k', 'v_proj': 'attn_v', 'out_proj': 'attn_output'}
            return f'blk.{layer_idx}.{proj_map.get(proj, proj)}.weight'
        elif 'mlp' in rest:
            proj = rest[rest.index('mlp') + 1] if 'mlp' in rest else ''
            proj_map = {'gate_proj': 'ffn_gate', 'up_proj': 'ffn_up', 'down_proj': 'ffn_down'}
            return f'blk.{layer_idx}.{proj_map.get(proj, proj)}.weight'
        elif 'input_layernorm' in rest:
            return f'blk.{layer_idx}.attn_norm.weight'
        elif 'post_attention_layernorm' in rest:
            return f'blk.{layer_idx}.ffn_norm.weight'
    elif 'lm_head' in parts:
        return 'output.weight'
    elif 'embed_tokens' in parts:
        return 'token_embd.weight'
    elif 'norm' in parts:
        return 'output_norm.weight'
    return name + '.weight'

def pack_q1_k(w):
    """Pack FP16 weights to Q1_K format."""
    w_flat = w.flatten().astype(np.float16)
    n = len(w_flat)
    n_blk = (n + Q1_K_ELEMS - 1) // Q1_K_ELEMS
    out = bytearray(n_blk * Q1_K_SIZE)

    for b in range(n_blk):
        s = b * Q1_K_ELEMS
        e = min(s + Q1_K_ELEMS, n)
        vals = w_flat[s:e]
        blk = b * Q1_K_SIZE

        vmin, vmax = float(vals.min()), float(vals.max())
        span = vmax - vmin
        struct.pack_into('<e', out, blk, np.float16(span))
        struct.pack_into('<e', out, blk + 2, np.float16(vmin))

        if span < 1e-9:
            for i in range(4): out[4 + i] = 0x88
            continue

        n_val = len(vals)
        for grp in range(4):
            s_idx, e_idx = grp * 64, min(grp * 64 + 64, n_val)
            grp_v = vals[s_idx:e_idx]
            gspan = float(grp_v.max()) - float(grp_v.min())
            if gspan < 1e-9:
                out[4 + grp] = 0x88
            else:
                target = gspan / 15.0
                best_nib, best_err = 8, abs(target - 1.0)
                for nib in range(16):
                    err = abs(target - (2.0 ** (nib - 8)))
                    if err < best_err:
                        best_err, best_nib = err, nib
                out[4 + grp] = (best_nib << 4) | best_nib

        for grp in range(4):
            s_idx, e_idx = grp * 64, min(grp * 64 + 64, n_val)
            grp_v = vals[s_idx:e_idx]
            nib = out[4 + grp] & 0x0F
            scale = 2.0 ** (nib - 8)
            base = float(vals[grp * 64]) if grp * 64 < n_val else 0.0
            for i, v in enumerate(grp_v):
                qv = max(0, min(15, round((float(v) - base) / scale)))
                row = i // 2
                if i % 2 == 0:
                    out[8 + grp * 32 + row] = int(qv) << 4
                else:
                    out[8 + grp * 32 + row] |= int(qv)
    return bytes(out)

import struct, numpy as np

GGUF_PATH = '/content/output/Qwen3.6-27B-Q1_K.gguf'

with open(GGUF_PATH, 'wb') as f:
    # Header
    f.write(struct.pack('<I', GGUF_MAGIC))
    f.write(struct.pack('<I', GGUF_VERSION))
    f.write(struct.pack('<Q', len(fp16_tensors)))
    f.write(struct.pack('<Q', 15))  # 15 KV pairs

    # Metadata
    tc = config.to_dict().get('text_config', config.to_dict())
    hpp = tc.get('hidden_size', config.hidden_size)
    nhead = tc.get('num_attention_heads', config.num_attention_heads)
    nkv = tc.get('num_key_value_heads', config.num_key_value_heads)
    nlayer = tc.get('num_hidden_layers', config.num_hidden_layers)
    inter = tc.get('intermediate_size', config.intermediate_size)
    vocab = tc.get('vocab_size', config.vocab_size)
    rope = tc.get('rope_theta', 10000.0)
    ctx = tc.get('max_position_embeddings', 32768)

    kv_pairs = [
        ('general.architecture', 'qwen2'),
        ('general.name', 'Qwen3.6-27B-FABQ-RC'),
        ('general.quantization_version', 2),
        ('general.file_type', 8),
        ('qwen2.block_count', nlayer),
        ('qwen2.embedding_length', hpp),
        ('qwen2.feed_forward_length', inter),
        ('qwen2.context_length', ctx),
        ('qwen2.attention.head_count', nhead),
        ('qwen2.attention.head_count_kv', nkv),
        ('qwen2.rope.dimension_count', hpp // nhead),
        ('qwen2.rope.freq_base', rope),
        ('qwen2.vocab_size', vocab),
        ('qwen2.attention.layer_norm_rms_epsilon', 1e-5),
        ('tokenizer.ggml.model', 'llama'),
    ]

    for key, val in kv_pairs:
        write_str(f, key)
        if isinstance(val, str):
            f.write(struct.pack('<I', GGUF_TYPE_STRING))
            write_str(f, val)
        elif isinstance(val, float):
            f.write(struct.pack('<I', GGUF_TYPE_FLOAT32))
            f.write(struct.pack('<f', val))
        else:
            f.write(struct.pack('<I', GGUF_TYPE_UINT32))
            f.write(struct.pack('<I', val))

    # Tensor info
    tensor_meta = []
    offset = 0
    for name, tensor in fp16_tensors:
        gguf_name = map_name(name)
        shape = list(tensor.shape)
        nbytes = tensor.nbytes

        write_str(f, gguf_name)
        f.write(struct.pack('<I', 2))  # 2D
        f.write(struct.pack('<Q', shape[0]))
        f.write(struct.pack('<Q', shape[1]))
        f.write(struct.pack('<I', GGML_TYPE_F16))
        f.write(struct.pack('<Q', offset))
        tensor_meta.append((gguf_name, tensor, shape, nbytes))
        offset += align_to(nbytes, ALIGN)

    # Padding
    f.write(b'\x00' * (align_to(f.tell(), ALIGN) - f.tell()))

    # Tensor data
    for name, tensor, shape, nbytes in tensor_meta:
        f.write(b'\x00' * (align_to(f.tell(), ALIGN) - f.tell()))
        f.write(tensor.cpu().numpy().astype(np.float16).tobytes())

print(f'GGUF written: {os.path.getsize(GGUF_PATH)/1e9:.2f} GB')

In [ ]:
# Cell 6: Also save .pth as backup
import shutil

pth_backup = '/content/output/quantized_model.pth'
shutil.copy(pth_path, pth_backup)
print(f'Copied .pth: {os.path.getsize(pth_backup)/1e9:.2f} GB')

In [ ]:
# Cell 7: Upload PRODUCED files to HuggingFace
from huggingface_hub import HfApi, create_repo

REPO_ID = 'toxzak/Qwen3.6-27B-FABQ-RC-GGUF'
api = HfApi(token=HF_TOKEN)

create_repo(repo_id=REPO_ID, token=HF_TOKEN, exist_ok=True, repo_type='model')
print(f'Repo: https://huggingface.co/{REPO_ID}')

# Upload produced files only
print('\nUploading PRODUCED files:')
for f in os.listdir('/content/output'):
    fpath = os.path.join('/content/output', f)
    size_gb = os.path.getsize(fpath) / 1e9
    print(f'  {f}: {size_gb:.2f} GB')
    t0 = time.time()
    api.upload_file(path_or_fileobj=fpath, path_in_repo=f, repo_id=REPO_ID, repo_type='model')
    print(f'    Uploaded in {time.time()-t0:.0f}s')

In [ ]:
# Cell 8: Create README
readme = '''---
language:
- en
license: apache-2.0
base_model: Qwen/Qwen3.6-27B
tags:
- quantization
- gguf
- llm
- fabq-rc
library_name: llama.cpp
---

# Qwen3.6-27B-FABQ-RC-GGUF

FABQ-RC quantized model in GGUF Q1_K format.

## Files

| File | Description |
|------|-------------|
| `Qwen3.6-27B-Q1_K.gguf` | FABQ-RC weights in GGUF Q1_K format |
| `quantized_model.pth` | Original FABQ-RC compressed weights |

## Use

```bash
./llama-cli -m Qwen3.6-27B-Q1_K.gguf -n 256 -p "Your prompt"
```

## Citation

```bibtex
@misc{fabqrc2026,
    author = {Zach Maronek},
    title = {FABQ-RC: Fisher-Adaptive Binary Quantization with Residual Codebooks},
    year = {2026},
    url = {https://github.com/toxzak/fabq-rc}
}
```
'''

readme_path = '/content/output/README.md'
with open(readme_path, 'w') as f:
    f.write(readme)

api.upload_file(path_or_fileobj=readme_path, path_in_repo='README.md', repo_id=REPO_ID, repo_type='model')
print('README uploaded')

In [ ]:
# Cell 9: Done
print('='*60)
print('DONE!')
print('='*60)
print(f'Repo: https://huggingface.co/{REPO_ID}')
print()
print('Produced files uploaded:')
for f in os.listdir('/content/output'):
    print(f'  - {f}')